# Register Tiling GEMM — Bank Conflict Fix + Profiling

**使用前请确保：** Runtime → Change runtime type → GPU (T4)

## 修复内容

基于 ncu profiling 发现的两个关键问题：

### 问题 1: Shared Memory Bank Conflict (3.2-way, 50% wavefronts)

**根因**：`As[BM][BK]` 中 BK=8，行间距 = 8 floats = 32 bytes。
同一 warp 内线程访问 `As[ty*8+0..7][bk]`（连续行的同一列），
32 bytes 间距恰好映射到同一 bank → 4-way bank conflict。

**修复**：`As[BM][BK + PAD]`，加 padding 让行间距变为 (8+4)×4 = 48 bytes，
破坏 bank 对齐，消除 conflict。

### 问题 2: Global Store Uncoalesced (12.5% 利用率)

**根因**：写回 C 时，同一 warp 内线程的列地址间隔 TN=8，不连续。

**修复**：写回时按行遍历，每行内连续 TN 个元素一起写。
warp 内相邻 tx 线程写连续的 TN 段，形成 coalesced access。

In [1]:
# 检查 GPU 和 CUDA 版本
!nvidia-smi
!nvcc --version

Mon Apr  6 00:46:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%%writefile matmul_reg_tiling_v2.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

#define BM 128
#define BN 128
#define BK 8
#define TM 8
#define TN 8
#define PAD 4   // shared memory padding 消除 bank conflict

__global__ void MatMulRegTilingKernel_v2(const float* __restrict__ A,
                                         const float* __restrict__ B,
                                         float* __restrict__ C,
                                         int M, int N, int K)
{
    const int bx = blockIdx.x;
    const int by = blockIdx.y;
    const int tx = threadIdx.x;
    const int ty = threadIdx.y;

    const int tid = ty * blockDim.x + tx;
    const int numThreads = blockDim.x * blockDim.y;

    // FIX 1: As 加 padding，行间距从 BK=8 变为 BK+PAD=12
    // 原来: As[row][col] 地址 = row * 8 * 4 = row * 32 bytes → 每隔 8 bank 循环 → 4-way conflict
    // 现在: As[row][col] 地址 = row * 12 * 4 = row * 48 bytes → 每隔 12 bank 循环 → 无 conflict
    __shared__ float As[BM][BK + PAD];  // 128 × 12 (原来 128 × 8)
    __shared__ float Bs[BK][BN];        // Bs 的列数 128，warp 连续列访问，无 conflict

    float regC[TM][TN] = {0.0f};
    float regA[TM];
    float regB[TN];

    const int cRow = by * BM;
    const int cCol = bx * BN;

    const float* aPtr = A + cRow * K;
    const float* bPtr = B + cCol;

    const int loadARow = tid / BK;
    const int loadACol = tid % BK;
    const int strideA  = numThreads / BK;

    const int loadBRow = tid / BN;
    const int loadBCol = tid % BN;
    const int strideB  = numThreads / BN;

    for (int k = 0; k < K; k += BK) {
        for (int i = 0; i < BM; i += strideA) {
            As[loadARow + i][loadACol] = aPtr[(loadARow + i) * K + k + loadACol];
        }
        for (int i = 0; i < BK; i += strideB) {
            Bs[loadBRow + i][loadBCol] = bPtr[(k + loadBRow + i) * N + loadBCol];
        }

        __syncthreads();

        for (int bk = 0; bk < BK; ++bk) {
            for (int tm = 0; tm < TM; ++tm) {
                regA[tm] = As[ty * TM + tm][bk];  // padding 后此访问无 bank conflict
            }
            for (int tn = 0; tn < TN; ++tn) {
                regB[tn] = Bs[bk][tx * TN + tn];
            }
            for (int tm = 0; tm < TM; ++tm) {
                for (int tn = 0; tn < TN; ++tn) {
                    regC[tm][tn] += regA[tm] * regB[tn];
                }
            }
        }

        __syncthreads();
    }

    // FIX 2: 写回 C — 外层遍历 tm (行)，内层遍历 tn (列)
    // warp 内相邻 tx 线程: globalCol = cCol + tx*TN + tn
    // tx=0 写 col 0..7, tx=1 写 col 8..15, ... → 连续地址 → coalesced
    for (int tm = 0; tm < TM; ++tm) {
        int globalRow = cRow + ty * TM + tm;
        for (int tn = 0; tn < TN; ++tn) {
            int globalCol = cCol + tx * TN + tn;
            C[globalRow * N + globalCol] = regC[tm][tn];
        }
    }
}

// ========== 原版 kernel (用于对比) ==========
__global__ void MatMulRegTilingKernel_v1(const float* __restrict__ A,
                                         const float* __restrict__ B,
                                         float* __restrict__ C,
                                         int M, int N, int K)
{
    const int bx = blockIdx.x;
    const int by = blockIdx.y;
    const int tx = threadIdx.x;
    const int ty = threadIdx.y;
    const int tid = ty * blockDim.x + tx;
    const int numThreads = blockDim.x * blockDim.y;

    __shared__ float As[BM][BK];    // 无 padding
    __shared__ float Bs[BK][BN];

    float regC[TM][TN] = {0.0f};
    float regA[TM];
    float regB[TN];

    const int cRow = by * BM;
    const int cCol = bx * BN;
    const float* aPtr = A + cRow * K;
    const float* bPtr = B + cCol;

    const int loadARow = tid / BK;
    const int loadACol = tid % BK;
    const int strideA  = numThreads / BK;
    const int loadBRow = tid / BN;
    const int loadBCol = tid % BN;
    const int strideB  = numThreads / BN;

    for (int k = 0; k < K; k += BK) {
        for (int i = 0; i < BM; i += strideA)
            As[loadARow + i][loadACol] = aPtr[(loadARow + i) * K + k + loadACol];
        for (int i = 0; i < BK; i += strideB)
            Bs[loadBRow + i][loadBCol] = bPtr[(k + loadBRow + i) * N + loadBCol];

        __syncthreads();

        for (int bk = 0; bk < BK; ++bk) {
            for (int tm = 0; tm < TM; ++tm)
                regA[tm] = As[ty * TM + tm][bk];
            for (int tn = 0; tn < TN; ++tn)
                regB[tn] = Bs[bk][tx * TN + tn];
            for (int tm = 0; tm < TM; ++tm)
                for (int tn = 0; tn < TN; ++tn)
                    regC[tm][tn] += regA[tm] * regB[tn];
        }
        __syncthreads();
    }

    for (int tm = 0; tm < TM; ++tm)
        for (int tn = 0; tn < TN; ++tn) {
            int globalRow = cRow + ty * TM + tm;
            int globalCol = cCol + tx * TN + tn;
            C[globalRow * N + globalCol] = regC[tm][tn];
        }
}

void MatMulCPU(const float* A, const float* B, float* C, int M, int N, int K)
{
    for (int i = 0; i < M; ++i)
        for (int j = 0; j < N; ++j) {
            float sum = 0.0f;
            for (int k = 0; k < K; ++k)
                sum += A[i * K + k] * B[k * N + j];
            C[i * N + j] = sum;
        }
}

int main()
{
    int M = 2048, N = 2048, K = 2048;

    size_t bytesA = M * K * sizeof(float);
    size_t bytesB = K * N * sizeof(float);
    size_t bytesC = M * N * sizeof(float);

    float* A     = (float*)malloc(bytesA);
    float* B     = (float*)malloc(bytesB);
    float* C_v1  = (float*)malloc(bytesC);
    float* C_v2  = (float*)malloc(bytesC);
    float* C_cpu = (float*)malloc(bytesC);

    for (int i = 0; i < M * K; ++i) A[i] = (float)(i % 10) * 0.1f;
    for (int i = 0; i < K * N; ++i) B[i] = (float)(i % 7)  * 0.1f;

    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, bytesA);
    cudaMalloc(&d_B, bytesB);
    cudaMalloc(&d_C, bytesC);
    cudaMemcpy(d_A, A, bytesA, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, B, bytesB, cudaMemcpyHostToDevice);

    dim3 dimBlock(BN / TN, BM / TM);  // (16, 16)
    dim3 dimGrid(N / BN, M / BM);     // (16, 16)

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);
    float ms = 0;

    // ========== v1: 原版 ==========
    MatMulRegTilingKernel_v1<<<dimGrid, dimBlock>>>(d_A, d_B, d_C, M, N, K);  // warmup
    cudaDeviceSynchronize();

    cudaEventRecord(start);
    MatMulRegTilingKernel_v1<<<dimGrid, dimBlock>>>(d_A, d_B, d_C, M, N, K);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    cudaEventElapsedTime(&ms, start, stop);
    float v1_ms = ms;
    cudaMemcpy(C_v1, d_C, bytesC, cudaMemcpyDeviceToHost);

    // ========== v2: bank conflict fix ==========
    MatMulRegTilingKernel_v2<<<dimGrid, dimBlock>>>(d_A, d_B, d_C, M, N, K);  // warmup
    cudaDeviceSynchronize();

    cudaEventRecord(start);
    MatMulRegTilingKernel_v2<<<dimGrid, dimBlock>>>(d_A, d_B, d_C, M, N, K);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    cudaEventElapsedTime(&ms, start, stop);
    float v2_ms = ms;
    cudaMemcpy(C_v2, d_C, bytesC, cudaMemcpyDeviceToHost);

    // ========== 验证 ==========
    float max_err = 0.0f;
    for (int i = 0; i < M * N; ++i) {
        float err = fabsf(C_v1[i] - C_v2[i]);
        if (err > max_err) max_err = err;
    }

    double gflops_v1 = 2.0 * M * N * K / (v1_ms * 1e-3) / 1e9;
    double gflops_v2 = 2.0 * M * N * K / (v2_ms * 1e-3) / 1e9;

    printf("=== Register Tiling GEMM: v1 vs v2 (bank conflict fix) ===\n");
    printf("Matrix: %dx%dx%d\n", M, N, K);
    printf("Block tile: %dx%d, Thread tile: %dx%d, K tile: %d\n", BM, BN, TM, TN, BK);
    printf("\n");
    printf("v1 (original):     %.3f ms  (%.1f GFLOPS)\n", v1_ms, gflops_v1);
    printf("v2 (padding fix):  %.3f ms  (%.1f GFLOPS)\n", v2_ms, gflops_v2);
    printf("Speedup: %.2fx\n", v1_ms / v2_ms);
    printf("Max diff (v1 vs v2): %e\n", max_err);
    if (max_err < 1e-3f)
        printf("PASSED\n");
    else
        printf("FAILED\n");

    cudaEventDestroy(start); cudaEventDestroy(stop);
    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    free(A); free(B); free(C_v1); free(C_v2); free(C_cpu);
    return 0;
}

Writing matmul_reg_tiling_v2.cu


In [3]:
!nvcc -O3 -lineinfo -arch=sm_75 matmul_reg_tiling_v2.cu -o matmul_reg_tiling_v2
print("编译成功!")

编译成功!


In [5]:
# 运行对比测试
!./matmul_reg_tiling_v2

=== Register Tiling GEMM: v1 vs v2 (bank conflict fix) ===
Matrix: 2048x2048x2048
Block tile: 128x128, Thread tile: 8x8, K tile: 8

v1 (original):     8.350 ms  (2057.6 GFLOPS)
v2 (padding fix):  8.465 ms  (2029.5 GFLOPS)
Speedup: 0.99x
Max diff (v1 vs v2): 0.000000e+00
PASSED


---
## Part 1: Nsight Systems — 系统级 profile

In [6]:
# 挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')
!chmod -R +x /content/drive/MyDrive/cuda/nsys_tool

NSYS = "/content/drive/MyDrive/cuda/nsys_tool/target-linux-x64/nsys"
!{NSYS} --version

Mounted at /content/drive
NVIDIA Nsight Systems version 2024.5.1.113-245134619542v0


In [10]:
NSYS = "/content/drive/MyDrive/cuda/nsys_tool/target-linux-x64/nsys"
!{NSYS} profile \
     --stats=true \
     --force-overwrite=true \
     -o nsys_reg_tiling_v2.qdrep \
     ./matmul_reg_tiling_v2

=== Register Tiling GEMM: v1 vs v2 (bank conflict fix) ===
Matrix: 2048x2048x2048
Block tile: 128x128, Thread tile: 8x8, K tile: 8

v1 (original):     12.961 ms  (1325.5 GFLOPS)
v2 (padding fix):  13.099 ms  (1311.5 GFLOPS)
Speedup: 0.99x
Max diff (v1 vs v2): 0.000000e+00
PASSED
Generating '/tmp/nsys-report-5876.qdstrm'
[1/8] [========================100%] nsys_reg_tiling_v2.nsys-rep
[2/8] [========================100%] nsys_reg_tiling_v2.sqlite
[3/8] Executing 'nvtx_sum' stats report
SKIPPED: /content/nsys_reg_tiling_v2.sqlite does not contain NV Tools Extension (NVTX) data.
[4/8] Executing 'osrt_sum' stats report

 Time (%)  Total Time (ns)  Num Calls    Avg (ns)     Med (ns)    Min (ns)   Max (ns)    StdDev (ns)            Name         
 --------  ---------------  ---------  ------------  -----------  --------  -----------  ------------  ----------------------
     75.7      528,958,880         14  37,782,777.1  9,289,356.5     1,334  328,103,479  86,059,075.1  poll                 

In [12]:
NSYS = "/content/drive/MyDrive/cuda/nsys_tool/target-linux-x64/nsys"
!{NSYS} stats --force-export=true --report cuda_gpu_kern_sum nsys_reg_tiling_v2.nsys-rep

Generating SQLite file nsys_reg_tiling_v2.sqlite from nsys_reg_tiling_v2.nsys-rep
Processing [nsys_reg_tiling_v2.sqlite] with [/content/drive/MyDrive/cuda/nsys_tool/host-linux-x64/reports/cuda_gpu_kern_sum.py]... 

 ** CUDA GPU Kernel Summary (cuda_gpu_kern_sum):

 Time (%)  Total Time (ns)  Instances    Avg (ns)      Med (ns)     Min (ns)    Max (ns)   StdDev (ns)                                       Name                                     
 --------  ---------------  ---------  ------------  ------------  ----------  ----------  -----------  ------------------------------------------------------------------------------
     50.3       26,166,505          2  13,083,252.5  13,083,252.5  13,079,844  13,086,661      4,820.3  MatMulRegTilingKernel_v2(const float *, const float *, float *, int, int, int)
     49.7       25,881,197          2  12,940,598.5  12,940,598.5  12,939,622  12,941,575      1,381.0  MatMulRegTilingKernel_v1(const float *, const float *, float *, int, int, int)



---
## Part 2: Nsight Compute — v2 kernel 深度分析

重点关注：
1. bank conflict 是否消除
2. MIO stall 是否降低
3. global store coalescing 是否改善

In [13]:
# Memory Workload — 重点看 bank conflict 是否消除
!ncu --section MemoryWorkloadAnalysis \
     --section MemoryWorkloadAnalysis_Tables \
     --kernel-name MatMulRegTilingKernel_v2 \
     --launch-skip 1 --launch-count 1 \
     ./matmul_reg_tiling_v2

==PROF== Connected to process 11212 (/content/matmul_reg_tiling_v2)
==PROF== Profiling "MatMulRegTilingKernel_v2": 0%....50%....100% - 27 passes
=== Register Tiling GEMM: v1 vs v2 (bank conflict fix) ===
Matrix: 2048x2048x2048
Block tile: 128x128, Thread tile: 8x8, K tile: 8

v1 (original):     14.866 ms  (1155.7 GFLOPS)
v2 (padding fix):  5933.781 ms  (2.9 GFLOPS)
Speedup: 0.00x
Max diff (v1 vs v2): 0.000000e+00
PASSED
==PROF== Disconnected from process 11212
[11212] matmul_reg_tiling_v2@127.0.0.1
  MatMulRegTilingKernel_v2(const float *, const float *, float *, int, int, int) (16, 16, 1)x(16, 16, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: Memory Workload Analysis
    ----------------- ----------- ------------
    Metric Name       Metric Unit Metric Value
    ----------------- ----------- ------------
    Memory Throughput     Gbyte/s        14.39
    Mem Busy                    %        43.15
    Max Bandwidth               %        28.57
    L1/TEX Hit Rate             

In [14]:
# Occupancy — 查看 padding 后 shared mem 增加对 occupancy 的影响
!ncu --section Occupancy \
     --section LaunchStats \
     --kernel-name MatMulRegTilingKernel_v2 \
     --launch-skip 1 --launch-count 1 \
     ./matmul_reg_tiling_v2

==PROF== Connected to process 11414 (/content/matmul_reg_tiling_v2)
==PROF== Profiling "MatMulRegTilingKernel_v2": 0%....50%....100% - 1 pass
=== Register Tiling GEMM: v1 vs v2 (bank conflict fix) ===
Matrix: 2048x2048x2048
Block tile: 128x128, Thread tile: 8x8, K tile: 8

v1 (original):     12.978 ms  (1323.8 GFLOPS)
v2 (padding fix):  830.395 ms  (20.7 GFLOPS)
Speedup: 0.02x
Max diff (v1 vs v2): 0.000000e+00
PASSED
==PROF== Disconnected from process 11414
[11414] matmul_reg_tiling_v2@127.0.0.1
  MatMulRegTilingKernel_v2(const float *, const float *, float *, int, int, int) (16, 16, 1)x(16, 16, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: Launch Statistics
    -------------------------------- --------------- ---------------
    Metric Name                          Metric Unit    Metric Value
    -------------------------------- --------------- ---------------
    Block Size                                                   256
    Function Cache Configuration                

In [15]:
# Scheduler + Warp Stall — MIO stall 是否降低
!ncu --section SchedulerStats \
     --section WarpStateStats \
     --kernel-name MatMulRegTilingKernel_v2 \
     --launch-skip 1 --launch-count 1 \
     ./matmul_reg_tiling_v2

==PROF== Connected to process 11500 (/content/matmul_reg_tiling_v2)
==PROF== Profiling "MatMulRegTilingKernel_v2": 0%....50%....100% - 8 passes
=== Register Tiling GEMM: v1 vs v2 (bank conflict fix) ===
Matrix: 2048x2048x2048
Block tile: 128x128, Thread tile: 8x8, K tile: 8

v1 (original):     9.314 ms  (1844.4 GFLOPS)
v2 (padding fix):  2473.037 ms  (6.9 GFLOPS)
Speedup: 0.00x
Max diff (v1 vs v2): 0.000000e+00
PASSED
==PROF== Disconnected from process 11500
[11500] matmul_reg_tiling_v2@127.0.0.1
  MatMulRegTilingKernel_v2(const float *, const float *, float *, int, int, int) (16, 16, 1)x(16, 16, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: Scheduler Statistics
    ---------------------------- ----------- ------------
    Metric Name                  Metric Unit Metric Value
    ---------------------------- ----------- ------------
    One or More Eligible                   %        30.86
    Issued Warp Per Scheduler                        0.31
    No Eligible               

In [16]:
# Compute + Speed of Light — 整体性能对比
!ncu --section SpeedOfLight \
     --section ComputeWorkloadAnalysis \
     --kernel-name MatMulRegTilingKernel_v2 \
     --launch-skip 1 --launch-count 1 \
     ./matmul_reg_tiling_v2

==PROF== Connected to process 11584 (/content/matmul_reg_tiling_v2)
==PROF== Profiling "MatMulRegTilingKernel_v2": 0%....50%....100% - 8 passes
=== Register Tiling GEMM: v1 vs v2 (bank conflict fix) ===
Matrix: 2048x2048x2048
Block tile: 128x128, Thread tile: 8x8, K tile: 8

v1 (original):     14.225 ms  (1207.8 GFLOPS)
v2 (padding fix):  2500.374 ms  (6.9 GFLOPS)
Speedup: 0.01x
Max diff (v1 vs v2): 0.000000e+00
PASSED
==PROF== Disconnected from process 11584
[11584] matmul_reg_tiling_v2@127.0.0.1
  MatMulRegTilingKernel_v2(const float *, const float *, float *, int, int, int) (16, 16, 1)x(16, 16, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         5.00
    SM Frequency                    Mhz       585.00
    Elapsed Cycles                cycle    8,

In [17]:
# Source Counters — 查看 uncoalesced access 是否改善
!ncu --section SourceCounters \
     --kernel-name MatMulRegTilingKernel_v2 \
     --launch-skip 1 --launch-count 1 \
     ./matmul_reg_tiling_v2

==PROF== Connected to process 11661 (/content/matmul_reg_tiling_v2)
==PROF== Profiling "MatMulRegTilingKernel_v2": 0%....50%....100% - 5 passes
=== Register Tiling GEMM: v1 vs v2 (bank conflict fix) ===
Matrix: 2048x2048x2048
Block tile: 128x128, Thread tile: 8x8, K tile: 8

v1 (original):     12.784 ms  (1343.9 GFLOPS)
v2 (padding fix):  2287.177 ms  (7.5 GFLOPS)
Speedup: 0.01x
Max diff (v1 vs v2): 0.000000e+00
PASSED
==PROF== Disconnected from process 11661
[11661] matmul_reg_tiling_v2@127.0.0.1
  MatMulRegTilingKernel_v2(const float *, const float *, float *, int, int, int) (16, 16, 1)x(16, 16, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: Source Counters
    ------------------------- ----------- ------------
    Metric Name               Metric Unit Metric Value
    ------------------------- ----------- ------------
    Branch Instructions Ratio           %         0.03
    Branch Instructions              inst   12,062,720
    Branch Efficiency                   %        

In [18]:
# 导出完整 ncu-rep 报告
!ncu --set full \
     --kernel-name MatMulRegTilingKernel_v2 \
     --launch-skip 1 --launch-count 1 \
     -o reg_tiling_v2_profile \
     ./matmul_reg_tiling_v2

print("\n报告已保存为 reg_tiling_v2_profile.ncu-rep")

==PROF== Connected to process 11755 (/content/matmul_reg_tiling_v2)
==PROF== Profiling "MatMulRegTilingKernel_v2": 0%....50%....100% - 31 passes
=== Register Tiling GEMM: v1 vs v2 (bank conflict fix) ===
Matrix: 2048x2048x2048
Block tile: 128x128, Thread tile: 8x8, K tile: 8

v1 (original):     13.625 ms  (1261.0 GFLOPS)
v2 (padding fix):  8503.721 ms  (2.0 GFLOPS)
Speedup: 0.00x
Max diff (v1 vs v2): 0.000000e+00
PASSED
==PROF== Disconnected from process 11755
==PROF== Report: /content/reg_tiling_v2_profile.ncu-rep

报告已保存为 reg_tiling_v2_profile.ncu-rep


In [19]:
# 下载报告文件
from google.colab import files
files.download('reg_tiling_v2_profile.ncu-rep')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## 对比检查清单

运行完 ncu 后，对比 v1 和 v2 的关键指标：

| 指标 | v1 (原版) | v2 (padding fix) | 改善? |
|------|-----------|-------------------|-------|
| Kernel 时间 (ms) | | | |
| GFLOPS | | | |
| Bank conflict wavefronts | 50% | | |
| MIO stall cycles | 7.0 / 12.2 (57.6%) | | |
| Eligible warps / scheduler | 0.42 | | |
| No Eligible % | 68.71% | | |
| Global store sectors utilized | 4.0 / 32 | | |
| Compute (SM) Throughput | 40.68% | | |
| Shared Mem Per Block (KB) | 8.19 | | |
| Occupancy (theoretical) | 50% | | |